# PS- 3B: ECG Anomaly Detection

ECG replaces the earlier MVTec dataset. In this ECG data, label 1 is normal and label 0 is abnormal.


In [6]:
# Import all packages first
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from tensorflow.keras import models, layers


In [7]:
# 1. Load updated ECG CSV. The last column is the label.
data = pd.read_csv("ecg.csv", header=None)
x = data.iloc[:, :-1]
y = data.iloc[:, -1].astype(int)
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)
normal = x_train[y_train == 0]
scaler = StandardScaler()
normal = scaler.fit_transform(normal)
x_test = scaler.transform(x_test)
x_train, x_val = train_test_split(normal, test_size=0.1, random_state=42)


In [8]:
# 2. Same autoencoder structure for every PS-3 dataset.
encoder = models.Sequential([
    layers.Input(shape=(x_train.shape[1],)),
    layers.Dense(20, activation="relu"),
    layers.Dense(14, activation="relu"),
    layers.Dense(8, activation="relu"),
    layers.Dense(4, activation="relu")
])
decoder = models.Sequential([
    layers.Input(shape=(4,)),
    layers.Dense(8, activation="relu"),
    layers.Dense(14, activation="relu"),
    layers.Dense(20, activation="relu"),
    layers.Dense(x_train.shape[1], activation="linear")
])
model = models.Sequential([encoder, decoder])
model.compile(optimizer="adam", loss="mse", metrics=["mae"])


In [9]:
# 3. Train only on normal data.
model.fit(x_train, x_train, validation_data=(x_val, x_val),
          epochs=20, batch_size=512, shuffle=True, verbose=2)


Epoch 1/20
3/3 - 7s - 2s/step - loss: 0.9952 - mae: 0.6922 - val_loss: 1.0410 - val_mae: 0.7252
Epoch 2/20
3/3 - 0s - 93ms/step - loss: 0.9942 - mae: 0.6917 - val_loss: 1.0396 - val_mae: 0.7246
Epoch 3/20
3/3 - 0s - 89ms/step - loss: 0.9925 - mae: 0.6910 - val_loss: 1.0366 - val_mae: 0.7234
Epoch 4/20
3/3 - 0s - 77ms/step - loss: 0.9895 - mae: 0.6898 - val_loss: 1.0311 - val_mae: 0.7214
Epoch 5/20
3/3 - 0s - 88ms/step - loss: 0.9845 - mae: 0.6880 - val_loss: 1.0229 - val_mae: 0.7184
Epoch 6/20
3/3 - 0s - 87ms/step - loss: 0.9780 - mae: 0.6854 - val_loss: 1.0119 - val_mae: 0.7142
Epoch 7/20
3/3 - 0s - 83ms/step - loss: 0.9688 - mae: 0.6818 - val_loss: 0.9970 - val_mae: 0.7084
Epoch 8/20
3/3 - 0s - 86ms/step - loss: 0.9564 - mae: 0.6767 - val_loss: 0.9777 - val_mae: 0.7003
Epoch 9/20
3/3 - 0s - 74ms/step - loss: 0.9413 - mae: 0.6703 - val_loss: 0.9553 - val_mae: 0.6903
Epoch 10/20
3/3 - 0s - 87ms/step - loss: 0.9234 - mae: 0.6625 - val_loss: 0.9333 - val_mae: 0.6799
Epoch 11/20
3/3 - 0s 

In [10]:
# 4. High reconstruction error means abnormal ECG.
threshold = np.percentile(np.mean((x_val - model.predict(x_val, verbose=0)) ** 2, axis=1), 99)
error = np.mean((x_test - model.predict(x_test, verbose=0)) ** 2, axis=1)
prediction = (error > threshold).astype(int)
y_test_anomaly = (y_test == 0).astype(int)
print("Threshold:", threshold)
print(classification_report(y_test_anomaly, prediction, target_names=["Normal", "Abnormal"], zero_division=0))

print("Number of anomalies detected:", prediction.sum())


Threshold: 8.711538947400566
              precision    recall  f1-score   support

      Normal       0.59      1.00      0.74       584
    Abnormal       1.00      0.01      0.01       416

    accuracy                           0.59      1000
   macro avg       0.79      0.50      0.38      1000
weighted avg       0.76      0.59      0.44      1000

